In [ ]:
#loading datasets
import pandas as pd

links = pd.read_csv("/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/raw/ml-32m/links.csv")
movies = pd.read_csv("/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/raw/ml-32m/movies.csv")
ratings = pd.read_csv("/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/raw/ml-32m/ratings.csv")
tags = pd.read_csv("/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/raw/ml-32m/tags.csv")

In [ ]:
#creating copies first
links_clean = links.copy()
movies_clean = movies.copy()
ratings_clean = ratings.copy()
tags_clean = tags.copy()

In [19]:
#handling missing values 

links_clean.isnull().sum()
links_clean["tmdbId"].isnull().sum()
links_clean["tmdbId"] = links_clean['tmdbId'].fillna(links_clean['tmdbId'].mean())
links_clean.duplicated().sum()
links_clean.isnull().sum()


movieId    0
imdbId     0
tmdbId     0
dtype: int64

In [16]:
movies_clean.isnull().sum()
movies_clean.duplicated().sum()


np.int64(0)

In [20]:
ratings_clean.isnull().sum()
ratings_clean.duplicated().sum()

np.int64(0)

In [37]:
print(tags_clean.isnull().sum())
print(tags_clean["tag"].isnull().sum())
tags_clean["tag"] = tags_clean["tag"].fillna("No Tags")
print(tags_clean.isnull().sum())
tags_clean.duplicated().sum()



userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
0
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


np.int64(0)

In [73]:
# Keep only required columns first
ratings_clean = ratings_clean[
    ["userId", "movieId", "rating", "timestamp", "date & time"]
]

# Create movie statistics
movie_stats = (
    ratings_clean
    .groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

# Create popularity score
movie_stats["popularity_score"] = (
    movie_stats["avg_rating"]
    * movie_stats["rating_count"]
)

# Merge all movie statistics into ratings_clean
ratings_clean = ratings_clean.merge(
    movie_stats,
    on="movieId",
    how="left"
)

# Display result
ratings_clean.head()

,userId,movieId,rating,timestamp,date & time,avg_rating,rating_count,popularity_score
0,1,17,4.0,944249077,1999-12-03 19:24:37,3.945126,22251,87783.0
1,1,25,1.0,944250228,1999-12-03 19:43:48,3.676182,22525,82806.0
2,1,29,2.0,943230976,1999-11-22 00:36:16,3.929884,9413,36992.0
3,1,30,5.0,944249077,1999-12-03 19:24:37,3.629615,1300,4718.5
4,1,32,5.0,943228858,1999-11-22 00:00:58,3.910185,55275,216135.5


In [54]:
tags_clean = tags.copy()
tags_clean.insert(
    4,
    "date & time",
    pd.to_datetime(
        tags_clean["timestamp"],
        unit="s"
    )
)
tags_clean["tag_length"] = (
    tags_clean["tag"]
    .str.len()
)
tags_clean.head()


,userId,movieId,tag,timestamp,date & time,tag_length
0,22,26479,Kevin Kline,1583038886,2020-03-01 05:01:26,11.0
1,22,79592,misogyny,1581476297,2020-02-12 02:58:17,8.0
2,22,247150,acrophobia,1622483469,2021-05-31 17:51:09,10.0
3,34,2174,music,1249808064,2009-08-09 08:54:24,5.0
4,34,2174,weird,1249808102,2009-08-09 08:55:02,5.0


In [53]:
movies_clean["year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)")
)
movies_clean["year"] = pd.to_numeric(
    movies_clean["year"],
    errors="coerce"
)
movies_clean["genre_list"] = (
    movies_clean["genres"]
    .str.split("|")
)
movies_clean.head()

,movieId,title,genres,year,genre_list
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),Comedy,1995.0,[Comedy]


,userId,movieId,rating,timestamp,date & time,rating_count_x,avg_rating,rating_count_y
0,1,17,4.0,944249077,1999-12-03 19:24:37,141,3.945126,22251
1,1,25,1.0,944250228,1999-12-03 19:43:48,141,3.676182,22525
2,1,29,2.0,943230976,1999-11-22 00:36:16,141,3.929884,9413
3,1,30,5.0,944249077,1999-12-03 19:24:37,141,3.629615,1300
4,1,32,5.0,943228858,1999-11-22 00:00:58,141,3.910185,55275


In [75]:
# now saving modified csv files 
links_clean.to_csv(
    "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/processed/links_clean.csv",
    index=False
)


In [76]:
movies_clean.to_csv(
    "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/processed/movies_clean.csv",
    index=False
)
ratings_clean.to_csv(
    "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/processed/ratingss_clean.csv",
    index=False
)
tags_clean.to_csv(
    "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/recommendation-engine-platform/data/processed/tags_clean.csv",
    index=False
)